# Scientific Image Forgery Detection
### Dual-Task Learning: Binary Classification + Copy-Move Localisation

**Architecture:** EfficientNet-B4 encoder · CBAM-UNet decoder · Deep supervision  
**Tasks:**
- Task 1 — Binary classification: authentic vs. forged  
- Task 2 — Pixel-level segmentation: copy-move region localisation  

**Key design choices:**
| Component | Choice | Rationale |
|-----------|--------|-----------|
| Encoder | EfficientNet-B4 | Strong ImageNet features; fits T4/P100 VRAM |
| Attention | CBAM (channel + spatial) | Highlights discriminative forgery cues |
| Deep supervision | Aux heads at d4, d3 | Forces intermediate decoder layers to localise |
| Cls loss | Focal (α=0.5, γ=2) | Down-weights easy negatives; balanced classes |
| Seg loss | BCE + Tversky (β=0.7) | Penalises false negatives more than FP |
| Class balance | WeightedRandomSampler | Ensures equal authentic/forged mini-batches |
| LR schedule | Warmup (5 ep) + CosineAnnealing | Stable early training, smooth decay |
| Composite metric | 0.4·AUC + 0.3·FgRecall + 0.3·Dice | Joint optimisation target for early stopping |

## 1. Environment Setup

In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'albumentations>=1.4,<2.0', 'timm>=0.9', 'seaborn'],
    check=True
)

## 2. Imports

In [ ]:
import os, random, warnings, time
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.cuda.amp import GradScaler, autocast
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import (
    roc_auc_score, accuracy_score, confusion_matrix,
    f1_score, roc_curve, classification_report,
)
from scipy.ndimage import label as scipy_label
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch : {torch.__version__}')
print(f'Device  : {DEVICE}')
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print(f'GPU     : {torch.cuda.get_device_name(0)}  (sm_{cap[0]}{cap[1]})')

## 3. Configuration

All hyperparameters in one place for reproducibility.

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
INPUT_DIR   = '/kaggle/input/datasets/llkh0a/recod-ailuc-scientific-image-forgery-detection'
TRAIN_IMGS  = f'{INPUT_DIR}/train_images'
TRAIN_MASKS = f'{INPUT_DIR}/train_masks'
SUPP_IMGS   = f'{INPUT_DIR}/supplemental_images'
SUPP_MASKS  = f'{INPUT_DIR}/supplemental_masks'
TEST_IMGS   = f'{INPUT_DIR}/test_images'
CKPT_BEST   = 'best_model.pt'
CKPT_FINAL  = 'final_model_v3.pt'

# ── Model ────────────────────────────────────────────────────────────────────
BACKBONE   = 'efficientnet_b4'   # b4 fits T4/P100; upgrade to b5/b7 on A100
IMG_SIZE   = 512

# ── Training ─────────────────────────────────────────────────────────────────
BATCH_SIZE = 4
MAX_EPOCHS = 60
LR         = 3e-4
CLIP_NORM  = 1.0
PATIENCE   = 12
VAL_FRAC   = 0.15
SEED       = 42

# ── Loss weights ─────────────────────────────────────────────────────────────
CLS_WEIGHT = 2.0   # classification loss multiplier
SEG_WEIGHT = 1.0   # segmentation loss multiplier
AUX_WEIGHT = 0.4   # deep-supervision auxiliary head weight

# ── ImageNet normalisation ────────────────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── Reproducibility ──────────────────────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

print('Configuration loaded.')

## 4. Dataset

### Dataset Structure
```
train_images/
    authentic/<uid>.png   # original, unmodified images
    forged/<uid>.png      # copy-move forged counterpart
train_masks/<uid>.npy     # binary ground-truth mask (0=background, 1=copied region)
supplemental_images/      # additional forged-only images
supplemental_masks/
test_images/              # held-out set (labels unknown)
```

### Split Strategy
- Splits are made at the **UID level** so authentic/forged pairs always land in the same fold.
- Supplemental images are always assigned to train (no authentic counterpart to pair).
- Split ratio: **70 % train / 15 % val** (15 % test omitted — competition test set is invalid).

In [ ]:
def load_mask(path: str, h: int, w: int) -> np.ndarray:
    """
    Load a .npy ground-truth mask.
    Handles multi-region masks [N,H,W] by taking the union (max).
    Resizes to (h, w) if stored at a different resolution.
    Returns a binary float32 array.
    """
    m = np.load(path)
    if m.ndim == 3:
        m = m.max(axis=0)                    # union over N regions
    if m.shape != (h, w):
        m = cv2.resize(m.astype(np.float32), (w, h),
                       interpolation=cv2.INTER_NEAREST)
    return (m > 0).astype(np.float32)


class ForgeryDataset(Dataset):
    """
    Each sample is a dict with keys:
        uid   : str   unique image identifier
        img   : str   path to PNG image
        label : int   0 = authentic, 1 = forged
        mask  : str | None   path to .npy mask (None for authentic)
    """

    def __init__(self, samples: list, transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        s   = self.samples[idx]
        img = cv2.cvtColor(cv2.imread(s['img']), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        mask = (load_mask(s['mask'], h, w)
                if s['label'] == 1 and s['mask']
                else np.zeros((h, w), dtype=np.float32))

        if self.transform:
            out  = self.transform(image=img, mask=mask)
            img  = out['image']        # torch.Tensor [C,H,W]
            mask = out['mask']         # torch.Tensor [H,W]

        return img, torch.tensor(s['label'], dtype=torch.float32), mask.unsqueeze(0)


def build_samples() -> tuple:
    """Scan dataset directories and return (all_samples, paired_uids)."""
    samples, uids = [], []

    auth_dir  = f'{TRAIN_IMGS}/authentic'
    forg_dir  = f'{TRAIN_IMGS}/forged'
    auth_uids = {os.path.splitext(f)[0] for f in os.listdir(auth_dir) if f.endswith('.png')}
    forg_uids = {os.path.splitext(f)[0] for f in os.listdir(forg_dir) if f.endswith('.png')}

    for uid in sorted(auth_uids & forg_uids):
        uids.append(uid)
        samples.append({'uid': uid, 'img': f'{auth_dir}/{uid}.png',
                        'label': 0, 'mask': None})
        mp = f'{TRAIN_MASKS}/{uid}.npy'
        if os.path.exists(mp):
            samples.append({'uid': uid, 'img': f'{forg_dir}/{uid}.png',
                            'label': 1, 'mask': mp})

    if os.path.isdir(SUPP_IMGS):
        for fname in sorted(os.listdir(SUPP_IMGS)):
            if not fname.endswith('.png'):
                continue
            uid = os.path.splitext(fname)[0]
            mp  = f'{SUPP_MASKS}/{uid}.npy'
            if os.path.exists(mp):
                samples.append({'uid': uid, 'img': f'{SUPP_IMGS}/{fname}',
                                'label': 1, 'mask': mp, 'supplemental': True})
    return samples, uids


# ── Build & split ─────────────────────────────────────────────────────────────
all_samples, all_uids = build_samples()

shuffled    = sorted(all_uids)
random.shuffle(shuffled)                      # reproducible via SEED
n_val       = int(VAL_FRAC * len(shuffled))
val_uids    = set(shuffled[:n_val])
train_uids  = set(shuffled[n_val:])

train_samples = [s for s in all_samples if s.get('supplemental') or s['uid'] in train_uids]
val_samples   = [s for s in all_samples if s['uid'] in val_uids]

n_tr_auth = sum(s['label'] == 0 for s in train_samples)
n_tr_forg = sum(s['label'] == 1 for s in train_samples)
n_va_auth = sum(s['label'] == 0 for s in val_samples)
n_va_forg = sum(s['label'] == 1 for s in val_samples)

print(f'Train : {len(train_samples):5d}  (auth={n_tr_auth}, forg={n_tr_forg})')
print(f'Val   : {len(val_samples):5d}  (auth={n_va_auth}, forg={n_va_forg})')
print(f'Auth/Forged ratio (train) : {n_tr_auth / max(n_tr_forg,1):.2f}')

## 5. Data Augmentation

**Training** uses aggressive spatial and colour augmentations that mimic common image manipulations (compression artefacts, noise, brightness shifts). All transforms are applied consistently to both the image and its mask.

**Validation / Test** only resize and normalise — no stochastic transforms.

In [ ]:
train_tf = A.Compose([
    A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.5, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10),
        A.CLAHE(clip_limit=2.0),
    ], p=0.6),
    A.OneOf([
        A.ImageCompression(quality_range=(70, 100)),
        A.GaussianBlur(blur_limit=(3, 5)),
        A.GaussNoise(std_range=(0.02, 0.1)),
    ], p=0.4),
    A.CoarseDropout(
        num_holes_range=(1, 4),
        hole_height_range=(16, 48),
        hole_width_range=(16, 48),
        fill=0, p=0.2,
    ),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

val_tf = A.Compose([
    A.Resize(height=IMG_SIZE, width=IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

test_tf = A.Compose([
    A.Resize(height=IMG_SIZE, width=IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

print('Transforms ready.')

## 6. DataLoaders

`WeightedRandomSampler` ensures each mini-batch contains an equal expected number of authentic and forged images, preventing the model from defaulting to the majority class.

In [ ]:
train_ds = ForgeryDataset(train_samples, train_tf)
val_ds   = ForgeryDataset(val_samples,   val_tf)

# Per-sample weight: 1 / class_frequency
train_labels   = [s['label'] for s in train_samples]
class_counts   = np.bincount(train_labels)
sample_weights = [1.0 / class_counts[l] for l in train_labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=2, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True,
)

print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')

## 7. Model Architecture

```
Input image [B, 3, 512, 512]
      │
      ▼
EfficientNet-B4 encoder (5 feature levels f0…f4)
      │
      ├─► Classification head:  GlobalAvgPool → FC(512) → Dropout → FC(1) → logit
      │
      └─► CBAM-UNet decoder:
            d4 (DecoderBlock + CBAM) ──► aux_head_d4 (deep supervision)
            d3 (DecoderBlock + CBAM) ──► aux_head_d3 (deep supervision)
            d2, d1, d0
            seg_head → full-resolution segmentation logit [B, 1, 512, 512]
```

**CBAM** applies channel-wise attention (squeeze-and-excitation style) followed by spatial attention (7×7 conv on avg+max channel maps), sharpening the decoder's focus on manipulated regions.

**Deep supervision** at d4 and d3 provides gradient signal directly to intermediate decoder layers, improving spatial localisation convergence.

In [ ]:
class CBAM(nn.Module):
    """Convolutional Block Attention Module — channel then spatial attention."""

    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        mid = max(channels // reduction, 4)
        self.channel_fc = nn.Sequential(
            nn.Linear(channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False),
        )
        self.spatial_conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c = x.shape[:2]
        # Channel attention
        avg = self.channel_fc(F.adaptive_avg_pool2d(x, 1).view(b, c)).view(b, c, 1, 1)
        mx  = self.channel_fc(F.adaptive_max_pool2d(x, 1).view(b, c)).view(b, c, 1, 1)
        x   = x * torch.sigmoid(avg + mx)
        # Spatial attention
        spatial = torch.cat([x.mean(1, keepdim=True), x.max(1, keepdim=True)[0]], dim=1)
        x = x * torch.sigmoid(self.spatial_conv(spatial))
        return x


class ConvBNReLU(nn.Sequential):
    def __init__(self, in_c: int, out_c: int, k: int = 3, p: int = 1):
        super().__init__(
            nn.Conv2d(in_c, out_c, k, padding=p, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )


class DecoderBlock(nn.Module):
    """Upsample → concat skip → 2× ConvBNReLU → CBAM."""

    def __init__(self, in_c: int, skip_c: int, out_c: int):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.conv = nn.Sequential(
            ConvBNReLU(in_c + skip_c, out_c),
            ConvBNReLU(out_c, out_c),
        )
        self.cbam = CBAM(out_c)

    def forward(self, x: torch.Tensor, skip: torch.Tensor = None) -> torch.Tensor:
        x = self.up(x)
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        return self.cbam(self.conv(x))


class EfficientUNet(nn.Module):
    """
    EfficientNet-B4 encoder + CBAM-UNet decoder with deep supervision.

    Returns
    -------
    cls_out  : [B, 1]          classification logit
    seg_out  : [B, 1, H, W]    full-resolution segmentation logit
    aux4_out : [B, 1, H/16, W/16]  deep supervision at d4
    aux3_out : [B, 1, H/8,  W/8 ]  deep supervision at d3
    """

    def __init__(self, backbone: str = BACKBONE, pretrained: bool = True):
        super().__init__()
        self.encoder = timm.create_model(
            backbone, pretrained=pretrained, features_only=True)
        ch = self.encoder.feature_info.channels()   # [ch0,ch1,ch2,ch3,ch4]

        self.d4 = DecoderBlock(ch[4], ch[3], 256)
        self.d3 = DecoderBlock(256,   ch[2], 128)
        self.d2 = DecoderBlock(128,   ch[1],  64)
        self.d1 = DecoderBlock( 64,   ch[0],  32)
        self.d0 = DecoderBlock( 32,       0,  16)

        self.seg_head  = nn.Conv2d(16,  1, 1)
        self.aux4_head = nn.Conv2d(256, 1, 1)
        self.aux3_head = nn.Conv2d(128, 1, 1)

        self.cls_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(ch[4], 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, 1),
        )

    def forward(self, x: torch.Tensor):
        f0, f1, f2, f3, f4 = self.encoder(x)
        cls_out = self.cls_head(f4)
        d4  = self.d4(f4, f3)
        d3  = self.d3(d4, f2)
        d2  = self.d2(d3, f1)
        d1  = self.d1(d2, f0)
        d0  = self.d0(d1)
        return cls_out, self.seg_head(d0), self.aux4_head(d4), self.aux3_head(d3)


# ── Instantiate ──────────────────────────────────────────────────────────────
model = EfficientUNet(backbone=BACKBONE, pretrained=True).to(DEVICE)
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params     : {total_params:,}')
print(f'Trainable params : {trainable_params:,}')

## 8. Loss Functions

### Classification — Focal Loss
Focal loss down-weights well-classified examples via the $(1-p_t)^\gamma$ modulating factor, focusing training on hard negatives. α=0.5 gives equal weight to both classes.

$$\mathcal{L}_{focal} = -\alpha_t (1-p_t)^\gamma \log(p_t)$$

### Segmentation — BCE + Tversky
Tversky loss generalises Dice loss with separate weights for false positives (α) and false negatives (β). Setting β > α biases the model toward completeness (not missing forged regions) at the cost of slight over-prediction.

$$\mathcal{L}_{tversky} = 1 - \frac{TP}{TP + \alpha\cdot FP + \beta\cdot FN}$$

### Multi-task Loss
$$\mathcal{L}_{total} = w_{cls}\cdot\mathcal{L}_{focal} + w_{seg}\cdot(\mathcal{L}_{seg} + w_{aux}\cdot(\mathcal{L}_{d4} + \mathcal{L}_{d3}))$$

In [ ]:
class FocalLoss(nn.Module):
    """Binary focal loss with configurable alpha and gamma."""

    def __init__(self, alpha: float = 0.5, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p   = torch.sigmoid(logits)
        pt  = targets * p + (1 - targets) * (1 - p)
        at  = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        return (at * (1 - pt) ** self.gamma * bce).mean()


def tversky_loss(
    logits: torch.Tensor,
    targets: torch.Tensor,
    alpha: float = 0.3,
    beta: float  = 0.7,
    eps: float   = 1e-7,
) -> torch.Tensor:
    """Tversky loss (beta > alpha penalises FN more than FP)."""
    p  = torch.sigmoid(logits)
    tp = (p * targets).sum()
    fp = (p * (1 - targets)).sum()
    fn = ((1 - p) * targets).sum()
    return 1 - (tp + eps) / (tp + alpha * fp + beta * fn + eps)


def seg_loss(pred: torch.Tensor, gt: torch.Tensor) -> torch.Tensor:
    return F.binary_cross_entropy_with_logits(pred, gt) + tversky_loss(pred, gt)


_focal = FocalLoss(alpha=0.5, gamma=2.0)


def multitask_loss(
    cls_out: torch.Tensor,
    seg_out: torch.Tensor,
    aux4:    torch.Tensor,
    aux3:    torch.Tensor,
    labels:  torch.Tensor,
    masks:   torch.Tensor,
) -> tuple:
    cls_l = _focal(cls_out.squeeze(1), labels)

    seg_l = torch.tensor(0.0, device=cls_out.device)
    forged = labels == 1
    if forged.any():
        gt_d4 = F.adaptive_avg_pool2d(masks[forged], aux4.shape[-2:])
        gt_d3 = F.adaptive_avg_pool2d(masks[forged], aux3.shape[-2:])
        seg_l = (
            seg_loss(seg_out[forged], masks[forged])
            + AUX_WEIGHT * seg_loss(aux4[forged], gt_d4)
            + AUX_WEIGHT * seg_loss(aux3[forged], gt_d3)
        )

    total = CLS_WEIGHT * cls_l + SEG_WEIGHT * seg_l
    return total, cls_l.detach(), seg_l.detach()


print('Loss functions ready.')

## 9. Optimiser & Learning Rate Schedule

**Differential learning rates:** the encoder (pretrained) is trained at 10× lower LR than the decoder (randomly initialised) to preserve ImageNet features early in training.

**Schedule:** 5-epoch linear warmup followed by cosine annealing decay to 1 % of peak LR.

In [ ]:
enc_params = list(model.encoder.parameters())
dec_params = [p for n, p in model.named_parameters() if not n.startswith('encoder')]

optimizer = torch.optim.AdamW(
    [
        {'params': enc_params, 'lr': LR * 0.1},
        {'params': dec_params, 'lr': LR},
    ],
    weight_decay=1e-4,
)

warmup = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=5)
cosine = CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS - 5, eta_min=LR * 0.01)
scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[5])

# AMP: only on Ampere+ (sm_80+). P100 = sm_60, T4 = sm_75, A100 = sm_80.
_sm = torch.cuda.get_device_capability(0)[0] if torch.cuda.is_available() else 0
USE_AMP = _sm >= 8
scaler  = GradScaler() if USE_AMP else None
print(f'AMP : {USE_AMP}  (sm_{_sm}x)')
print(f'Encoder LR : {LR * 0.1:.2e}  |  Decoder LR : {LR:.2e}')

## 10. Training & Validation Functions

**Composite validation metric** (used for early stopping and checkpoint selection):

$$\text{Comp} = 0.4 \cdot \text{AUC} + 0.3 \cdot \text{ForgeryRecall} + 0.3 \cdot \text{Dice}$$

This jointly rewards discrimination ability (AUC), sensitivity to forged images (Forged Recall), and spatial localisation accuracy (Dice).

In [ ]:
def train_one_epoch(model, loader, optimizer, epoch: int) -> tuple:
    model.train()
    losses, cls_ls, seg_ls = [], [], []
    pbar = tqdm(loader, desc=f'Epoch {epoch:02d} [Train]', leave=False)

    for imgs, labels, masks in pbar:
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        masks  = masks.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if USE_AMP:
            with autocast():
                cls_out, seg_out, aux4, aux3 = model(imgs)
                loss, cl, sl = multitask_loss(cls_out, seg_out, aux4, aux3, labels, masks)
        else:
            cls_out, seg_out, aux4, aux3 = model(imgs)
            loss, cl, sl = multitask_loss(cls_out, seg_out, aux4, aux3, labels, masks)

        if not torch.isfinite(loss):
            optimizer.zero_grad(set_to_none=True)
            continue

        if USE_AMP:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
            optimizer.step()

        losses.append(loss.item())
        cls_ls.append(cl.item())
        seg_ls.append(sl.item())
        pbar.set_postfix({
            'loss': f'{np.mean(losses):.4f}',
            'cls':  f'{np.mean(cls_ls):.4f}',
            'seg':  f'{np.mean(seg_ls):.4f}',
        })

    return (float(np.mean(losses)),
            float(np.mean(cls_ls)),
            float(np.mean(seg_ls)))


@torch.no_grad()
def validate(model, loader, threshold: float = 0.5) -> dict:
    model.train(False)
    y_true, y_prob, dice_scores = [], [], []

    for imgs, labels, masks in tqdm(loader, desc='Val', leave=False):
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        masks  = masks.to(DEVICE, non_blocking=True)

        cls_out, seg_out, _, _ = model(imgs)
        probs = torch.sigmoid(cls_out).squeeze(1)

        y_true.extend(labels.cpu().numpy())
        y_prob.extend(probs.cpu().numpy())

        for i in range(len(labels)):
            if labels[i].item() == 1:
                pred  = (torch.sigmoid(seg_out[i, 0]) > threshold).float()
                gt    = masks[i, 0]
                inter = (pred * gt).sum()
                dice_scores.append(
                    (2 * inter / (pred.sum() + gt.sum() + 1e-7)).item()
                )

    y_pred = (np.array(y_prob) > threshold).astype(int)
    acc    = accuracy_score(y_true, y_pred)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = 0.5
    dice  = float(np.mean(dice_scores)) if dice_scores else 0.0
    cm    = confusion_matrix(y_true, y_pred)
    fgrec = (cm[1, 1] / (cm[1, 0] + cm[1, 1] + 1e-7)
             if cm.shape == (2, 2) else 0.0)
    comp  = 0.4 * auc + 0.3 * fgrec + 0.3 * dice

    return {'acc': acc, 'auc': auc, 'dice': dice, 'fgrec': fgrec, 'comp': comp}


print('Train/validate functions ready.')

## 11. Training Loop

- Best checkpoint saved to `best_model.pt` whenever the composite metric improves.
- Early stopping triggers after `PATIENCE` consecutive epochs without improvement.
- All per-epoch metrics are logged to `history` for post-training analysis.

In [ ]:
history      = []
best_comp    = -float('inf')
patience_ctr = 0

for epoch in range(1, MAX_EPOCHS + 1):
    tr_loss, tr_cls, tr_seg = train_one_epoch(model, train_loader, optimizer, epoch)
    val_m = validate(model, val_loader)
    scheduler.step()

    row = {
        'epoch'  : epoch,
        'tr_loss': tr_loss, 'tr_cls': tr_cls, 'tr_seg': tr_seg,
        **{f'val_{k}': v for k, v in val_m.items()},
    }
    history.append(row)

    print(
        f'[{epoch:02d}/{MAX_EPOCHS}] '
        f'loss={tr_loss:.4f} cls={tr_cls:.4f} seg={tr_seg:.4f} | '
        f'Acc={val_m["acc"]:.4f} AUC={val_m["auc"]:.4f} '
        f'Dice={val_m["dice"]:.4f} FgRec={val_m["fgrec"]:.4f} '
        f'Comp={val_m["comp"]:.4f}'
    )

    if val_m['comp'] > best_comp:
        best_comp    = val_m['comp']
        patience_ctr = 0
        torch.save(model.state_dict(), CKPT_BEST)
        print(f'  --> Best checkpoint saved  (comp={best_comp:.4f})')
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'Early stopping triggered at epoch {epoch}.')
            break

print(f'\nTraining complete. Best composite score: {best_comp:.4f}')

## 12. Training Curves

In [ ]:
epochs   = [r['epoch']    for r in history]
tr_loss  = [r['tr_loss']  for r in history]
val_auc  = [r['val_auc']  for r in history]
val_dice = [r['val_dice'] for r in history]
val_comp = [r['val_comp'] for r in history]
val_fgr  = [r['val_fgrec'] for r in history]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Training History', fontsize=14, fontweight='bold')

axes[0, 0].plot(epochs, tr_loss, color='steelblue', lw=2)
axes[0, 0].set_title('Train Loss'); axes[0, 0].set_xlabel('Epoch')
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(epochs, val_auc, color='darkorange', lw=2)
axes[0, 1].set_title('Val AUC'); axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylim(0.4, 1.0); axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(epochs, val_dice, color='mediumseagreen', lw=2)
axes[1, 0].set_title('Val Dice (seg)'); axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylim(0, 1.0); axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(epochs, val_comp, color='mediumpurple', lw=2, label='Composite')
axes[1, 1].plot(epochs, val_fgr,  color='tomato',       lw=2, linestyle='--', label='FgRecall')
best_ep = epochs[int(np.argmax(val_comp))]
axes[1, 1].axvline(best_ep, color='k', linestyle=':', lw=1.2, label=f'Best ep={best_ep}')
axes[1, 1].set_title('Composite & Forged Recall'); axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylim(0, 1.0); axes[1, 1].legend(fontsize=9); axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)
print('Saved -> training_curves.png')

## 13. Post-processing & Test-Time Augmentation

**Post-processing pipeline:**
1. Threshold the probability map
2. Morphological closing (5×5 ellipse kernel) — fills small holes within regions
3. Remove connected components smaller than `min_area` pixels

**TTA:** 4 augmentation variants (original + 3 flips). Masks are un-flipped before averaging so spatial predictions align correctly.

In [ ]:
def postprocess_mask(prob_map: np.ndarray,
                     threshold: float = 0.5,
                     min_area: int    = 200) -> np.ndarray:
    """Binarise → morphological close → remove small components."""
    binary = (prob_map > threshold).astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    labeled, n = scipy_label(binary)
    out = np.zeros_like(binary)
    for i in range(1, n + 1):
        if (labeled == i).sum() >= min_area:
            out[labeled == i] = 1
    return out.astype(np.float32)


@torch.no_grad()
def tta_predict(model, img_tensor: torch.Tensor) -> tuple:
    """
    4-fold TTA (original + H-flip + V-flip + HV-flip).
    Masks are un-flipped before averaging.
    Returns (cls_prob: float, seg_prob: np.ndarray [H, W]).
    """
    model.train(False)
    cls_logits, seg_logits = [], []

    for hflip in (False, True):
        for vflip in (False, True):
            x = img_tensor.clone()
            if hflip: x = torch.flip(x, dims=[3])
            if vflip: x = torch.flip(x, dims=[2])

            c, s, _, _ = model(x)

            if hflip: s = torch.flip(s, dims=[3])
            if vflip: s = torch.flip(s, dims=[2])

            cls_logits.append(c)
            seg_logits.append(s)

    cls_prob = torch.sigmoid(torch.stack(cls_logits).mean(0)).item()
    seg_prob = torch.sigmoid(torch.stack(seg_logits).mean(0)).squeeze().cpu().numpy()
    return cls_prob, seg_prob


print('Post-processing and TTA ready.')

## 14. Final Evaluation on Validation Set

Load best checkpoint → sweep threshold for optimal F1 → compute all metrics (both Dice and classification metrics use the same threshold).

In [ ]:
# ── Load best checkpoint ─────────────────────────────────────────────────────
ckpt = torch.load(CKPT_BEST, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt)
model.train(False)
print(f'Loaded: {CKPT_BEST}')

# ── Collect predictions ───────────────────────────────────────────────────────
y_true, y_prob = [], []
seg_probs_all, masks_all = [], []

with torch.no_grad():
    for imgs, labels, masks in tqdm(val_loader, desc='Evaluating'):
        imgs   = imgs.to(DEVICE)
        labels = labels.to(DEVICE)
        masks  = masks.to(DEVICE)

        cls_out, seg_out, _, _ = model(imgs)
        probs     = torch.sigmoid(cls_out).squeeze(1).cpu().numpy()
        seg_probs = torch.sigmoid(seg_out).squeeze(1).cpu().numpy()
        mask_np   = masks.squeeze(1).numpy()

        y_true.extend(labels.cpu().numpy())
        y_prob.extend(probs)

        for i in range(len(labels)):
            if labels[i].item() == 1:
                seg_probs_all.append(seg_probs[i])
                masks_all.append(mask_np[i])

y_true = np.array(y_true)
y_prob = np.array(y_prob)

# ── Threshold sweep (maximise F1 over forged class) ───────────────────────────
best_thr, best_f1 = 0.5, 0.0
for thr in np.arange(0.05, 0.95, 0.01):
    preds = (y_prob > thr).astype(int)
    f1    = f1_score(y_true, preds, zero_division=0)
    if f1 > best_f1:
        best_f1, best_thr = f1, float(thr)

# ── Dice at best_thr (consistent with classification threshold) ───────────────
dice_list = []
for sp, gt in zip(seg_probs_all, masks_all):
    pred_bin = (sp > best_thr).astype(np.float32)
    inter    = (pred_bin * gt).sum()
    denom    = pred_bin.sum() + gt.sum() + 1e-7
    dice_list.append(2 * inter / denom)

y_pred = (y_prob > best_thr).astype(int)
cm     = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()
auc  = roc_auc_score(y_true, y_prob)
dice = float(np.mean(dice_list)) if dice_list else float('nan')

# ── Dashboard plots ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Validation Evaluation  —  Best Checkpoint', fontsize=14, fontweight='bold')

ax = axes[0]
ax.hist(y_prob[y_true == 0], bins=40, alpha=0.65, color='steelblue',
        label=f'Authentic  n={(y_true==0).sum()}')
ax.hist(y_prob[y_true == 1], bins=40, alpha=0.65, color='tomato',
        label=f'Forged     n={(y_true==1).sum()}')
ax.axvline(best_thr, color='k', lw=1.5, linestyle='--',
           label=f'threshold={best_thr:.2f}')
ax.set_xlabel('Predicted probability'); ax.set_ylabel('Count')
ax.set_title('Score Distribution'); ax.legend(fontsize=9)

ax = axes[1]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Authentic', 'Forged'],
            yticklabels=['Authentic', 'Forged'],
            annot_kws={'size': 16})
ax.set_xlabel('Predicted', fontsize=11); ax.set_ylabel('Actual', fontsize=11)
ax.set_title(f'Confusion Matrix  (thr={best_thr:.2f})')

ax = axes[2]
fpr, tpr, _ = roc_curve(y_true, y_prob)
ax.plot(fpr, tpr, lw=2.5, color='darkorange', label=f'AUC = {auc:.4f}')
ax.fill_between(fpr, tpr, alpha=0.10, color='darkorange')
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve'); ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('eval_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

# ── Printed metrics ───────────────────────────────────────────────────────────
prec = tp / (tp + fp + 1e-7)
rec  = tp / (tp + fn + 1e-7)
spec = tn / (tn + fp + 1e-7)
f1v  = 2 * tp / (2 * tp + fp + fn + 1e-7)

print('=' * 50)
print(f'  Threshold       : {best_thr:.2f}')
print(f'  Accuracy        : {accuracy_score(y_true, y_pred):.4f}')
print(f'  AUC             : {auc:.4f}')
print(f'  Precision       : {prec:.4f}')
print(f'  Forged Recall   : {rec:.4f}  (sensitivity)')
print(f'  Specificity     : {spec:.4f}  (authentic recall)')
print(f'  F1              : {f1v:.4f}')
print(f'  Mean Dice (seg) : {dice:.4f}')
print('=' * 50)
print()
print(classification_report(y_true, y_pred, target_names=['Authentic', 'Forged']))

## 15. Save Final Model Bundle

In [ ]:
torch.save(
    {
        'model_state_dict': model.state_dict(),
        'cls_threshold'   : best_thr,
        'seg_threshold'   : best_thr,
        'img_size'        : IMG_SIZE,
        'backbone'        : BACKBONE,
        'architecture'    : 'EfficientNet-B4-CBAM-UNet-DeepSupervision',
        'val_auc'         : float(auc),
        'val_dice'        : float(dice),
        'val_comp'        : float(best_comp),
    },
    CKPT_FINAL,
)
sz = os.path.getsize(CKPT_FINAL) / 1e6
print(f'Saved {CKPT_FINAL}  ({sz:.1f} MB)')
print(f'  AUC  : {auc:.4f}')
print(f'  Dice : {dice:.4f}')
print(f'  Comp : {best_comp:.4f}')

## 16. Inference

### `predict_image(image_path)`
- Runs 4-fold TTA
- Applies morphological post-processing to the mask
- Displays 3-panel output for forged images (original / red overlay / jet heatmap)
- Returns a result dict: `{label, cls_prob, mask}`

In [ ]:
import glob


def predict_image(image_path: str,
                  cls_thr: float    = None,
                  seg_thr: float    = None,
                  min_area: int     = 200,
                  save_prefix: str  = None) -> dict:
    """
    Run inference on a single image.

    Parameters
    ----------
    image_path  : path to PNG/JPG
    cls_thr     : classification threshold (default: best_thr from evaluation)
    seg_thr     : segmentation threshold   (default: best_thr)
    min_area    : minimum kept forgery region in pixels
    save_prefix : if set, saves output images to '<save_prefix>_*.png'

    Returns
    -------
    dict  label (str), cls_prob (float), mask (np.ndarray H x W)
    """
    if cls_thr is None: cls_thr = best_thr
    if seg_thr is None: seg_thr = best_thr

    orig_bgr       = cv2.imread(image_path)
    orig_rgb       = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB)
    orig_h, orig_w = orig_rgb.shape[:2]

    tensor             = test_tf(image=orig_rgb)['image'].unsqueeze(0).to(DEVICE)
    cls_prob, seg_prob = tta_predict(model, tensor)
    label              = 'Forged' if cls_prob > cls_thr else 'Authentic'

    seg_up   = cv2.resize(seg_prob, (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)
    bin_mask = postprocess_mask(seg_up, seg_thr, min_area)

    if label == 'Authentic':
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.imshow(orig_rgb)
        ax.set_title(f'AUTHENTIC  (confidence: {1 - cls_prob:.1%})', fontsize=13)
        ax.axis('off')
        plt.tight_layout()
        if save_prefix:
            plt.savefig(f'{save_prefix}_authentic.png', dpi=150, bbox_inches='tight')
        plt.show()
        plt.close(fig)
    else:
        overlay = orig_rgb.copy().astype(np.float32)
        overlay[bin_mask > 0] = (overlay[bin_mask > 0] * 0.45
                                 + np.array([255, 40, 40]) * 0.55)
        overlay = np.clip(overlay, 0, 255).astype(np.uint8)

        heatmap_rgb = cv2.cvtColor(
            cv2.applyColorMap((seg_up * 255).astype(np.uint8), cv2.COLORMAP_JET),
            cv2.COLOR_BGR2RGB)

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        axes[0].imshow(orig_rgb)
        axes[0].set_title('Input Image', fontsize=13)
        axes[1].imshow(overlay)
        axes[1].set_title(f'FORGED  (p={cls_prob:.3f})\nRed = copy-move region',
                          fontsize=11)
        axes[2].imshow(heatmap_rgb)
        axes[2].set_title('Segmentation Heatmap (jet)', fontsize=13)
        for ax in axes:
            ax.axis('off')
        plt.tight_layout()
        if save_prefix:
            plt.savefig(f'{save_prefix}_prediction.png', dpi=150, bbox_inches='tight')
        plt.show()
        plt.close(fig)

        fig2, ax2 = plt.subplots(figsize=(5, 5))
        ax2.imshow(bin_mask, cmap='gray', vmin=0, vmax=1)
        ax2.set_title(
            f'Binary Forgery Mask\n({bin_mask.mean() * 100:.1f}% of image)',
            fontsize=13)
        ax2.axis('off')
        plt.tight_layout()
        if save_prefix:
            plt.savefig(f'{save_prefix}_mask.png', dpi=150, bbox_inches='tight')
        plt.show()
        plt.close(fig2)

    print(f'Label    : {label}')
    print(f'Conf     : {cls_prob:.4f}')
    if label == 'Forged':
        print(f'Forged % : {bin_mask.mean() * 100:.2f}% of image')
    return {'label': label, 'cls_prob': cls_prob, 'mask': bin_mask}


# ── Choose image ──────────────────────────────────────────────────────────────
#
# Option A — paste a path directly:
#   IMAGE_PATH = '/kaggle/input/.../train_images/forged/123.png'
#
# Option B — pick by index from the listing below:

SOURCE = 'test'   # 'test' | 'forged' | 'authentic' | 'supplemental'
_dirs = {
    'test':         sorted(glob.glob(f'{TEST_IMGS}/*.png')),
    'forged':       sorted(glob.glob(f'{TRAIN_IMGS}/forged/*.png')),
    'authentic':    sorted(glob.glob(f'{TRAIN_IMGS}/authentic/*.png')),
    'supplemental': sorted(glob.glob(f'{SUPP_IMGS}/*.png')),
}
_files = _dirs.get(SOURCE, [])

print(f'Available [{SOURCE}] images ({len(_files)} total):')
for i, p in enumerate(_files[:20]):
    print(f'  [{i:3d}]  {os.path.basename(p)}')
if len(_files) > 20:
    print(f'  ... and {len(_files) - 20} more')

INDEX      = 0           # change to any index above
IMAGE_PATH = _files[INDEX] if _files else None

if IMAGE_PATH:
    print(f'\nRunning on: {IMAGE_PATH}')
    result = predict_image(IMAGE_PATH)
else:
    print('No images found for SOURCE =', SOURCE)